# NB_patient_360
Builds `adb_neeleshcaresync.gold.patient_360` from the silver layer per the STTM (`sttm_silver_to_gold.xlsx`, sheet `patient_360`).

* **Grain:** one row per patient
* **Driving table:** `silver.patients`
* **Joins (LEFT):** `silver.hospitals` on `registered_hospital_id = hospital_id`; `silver.insurance_providers` on `insurance_provider_id`; aggregated `silver.lab_results` grouped by `patient_id`
* **Filters:** none
* **Load type:** FULL — `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
# Catalog / schema configuration (STTM placeholder <catalog>)
catalog = "adb_neeleshcaresync"
silver = f"{catalog}.silver"
target_table = f"{catalog}.gold.patient_360"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per patient with hospital, insurance and lab test summary (STTM patient_360)'
AS
WITH lab_results_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                  AS total_tests,
    COUNT_IF(result_status = 'Abnormal')      AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')      AS critical_test_count,
    MAX(test_date)                            AS last_test_date
  FROM {silver}.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                              AS patient_id,
  CAST(p.first_name AS STRING)                                           AS first_name,
  CAST(p.last_name AS STRING)                                            AS last_name,
  CAST(p.date_of_birth AS DATE)                                          AS date_of_birth,
  CAST(FLOOR(DATEDIFF(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  CAST(p.gender AS STRING)                                               AS gender,
  CAST(p.city AS STRING)                                                 AS city,
  CAST(p.state AS STRING)                                                AS state,
  CAST(p.registered_hospital_id AS INT)                                  AS hospital_id,
  CAST(h.hospital_name AS STRING)                                        AS hospital_name,
  CAST(p.insurance_provider_id AS INT)                                   AS insurance_provider_id,
  CAST(ip.provider_name AS STRING)                                       AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                        AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                        AS critical_test_count,
  CAST(l.last_test_date AS DATE)                                         AS last_test_date
FROM {silver}.patients p
LEFT JOIN {silver}.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN {silver}.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_results_agg l
  ON p.patient_id = l.patient_id
""")
print(f"Created {target_table}")

In [0]:
# Validate grain and reconciliation against silver sources
src_patients = spark.table(f"{silver}.patients").count()
src_tests = spark.table(f"{silver}.lab_results").count()

stats = spark.sql(f"""
SELECT
  COUNT(*)                          AS row_count,
  COUNT(DISTINCT patient_id)        AS distinct_patients,
  SUM(total_tests)                  AS sum_tests,
  COUNT_IF(total_tests = 0 AND last_test_date IS NOT NULL) AS bad_last_test,
  COUNT_IF(abnormal_test_count + critical_test_count > total_tests) AS bad_counts
FROM {target_table}
""").first()

checks = {
    "One row per patient (row count = silver.patients)": stats.row_count == src_patients,
    "patient_id unique": stats.distinct_patients == stats.row_count,
    "SUM(total_tests) <= silver.lab_results rows": stats.sum_tests <= src_tests,
    "last_test_date NULL when no tests": stats.bad_last_test == 0,
    "abnormal + critical <= total_tests": stats.bad_counts == 0,
}
for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
print(stats.asDict())
assert all(checks.values()), "patient_360 validation failed"

In [0]:
display(spark.table(target_table).orderBy("patient_id").limit(100))

In [0]:
# Return the gold table record count to the caller (e.g. job / dbutils.notebook.run)
updated_df = spark.table(target_table)
record_count = updated_df.count()
print(f"{target_table} record count: {record_count}")
dbutils.notebook.exit(str(record_count))